# Filter Dataset by Required Wikipedia Languages

## Purpose
Filters the cross-verified notable entities database to retain only entities with Wikipedia articles in **all five languages**: en, it, fr, es, de.

## Research Focus
Analyze bias across three dimensions:
- **Gender**: Male vs Female representation
- **Geographic**: Western vs Non-Western regions (UN subregion)
- **Temporal**: Historical periods (bigperiod_birth)

## Data Source
- **Input**: `cross-verified-database.csv.gz` from [BHHT Datascape](https://medialab.github.io/bhht-datascape/)
- **Output**: `data/entities_filtered_by_languages.csv`

**Note**: Language selection focuses on Western Wikipedia editions by European speaker population. TODO: Discuss expanding to non-Western editions with professor.

## 1. Import Libraries

In [ ]:
import pandas as pd
from typing import List

## 2. Configuration

In [ ]:
# Configuration Constants
REQUIRED_LANGUAGES = ['en', 'it', 'fr', 'es', 'de']  
# Western Wikipedia editions sorted by European speaker population
# Source: https://meta.wikimedia.org/wiki/List_of_Wikipedias

INPUT_FILE = '../data/cross-verified-database.csv.gz'
OUTPUT_FILE = '../data/entities_filtered_by_languages.csv'

## 3. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv(INPUT_FILE, compression='gzip', encoding='latin-1')

# Rename wikidata_code to wikidata_id for consistency across the pipeline
df = df.rename(columns={'wikidata_code': 'wikidata_id'})

In [ ]:
# Display basic dataset information
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# Show all available columns
print("Available columns in the dataset:")
print("=" * 80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")
print("=" * 80)

### Examine Key Columns for Analysis
We'll inspect the columns that are relevant for bias analysis across three dimensions:
- **Identity Column**: `wikidata_id` - Unique identifier for each entity
- **Gender**: `gender` - Male/Female representation
- **Geographic**: `un_subregion` - Geographic regions based on UN classification
- **Temporal**: `bigperiod_birth` - Historical period bins, `birth` - Year of birth

In [ ]:
# Examine the key columns for our analysis
analysis_columns = ['wikidata_id', 'gender', 'bigperiod_birth', 'un_subregion']

print("Sample data from key columns:")
print("=" * 80)
print(df[analysis_columns].head(10))
print("\n")

In [ ]:
# Display unique values for categorical columns
print("GENDER - Unique values:")
print("-" * 80)
gender_counts = df['gender'].value_counts(dropna=False)
for value, count in gender_counts.items():
    print(f"  {value}: {count:,} ({count/len(df)*100:.1f}%)")

print("\n" + "=" * 80)
print("TEMPORAL (bigperiod_birth) - Unique values:")
print("-" * 80)
birth_periods = df['bigperiod_birth'].value_counts(dropna=False).sort_index()
for period, count in birth_periods.items():
    print(f"  {period}: {count:,} ({count/len(df)*100:.1f}%)")

print("\n" + "=" * 80)
print("GEOGRAPHIC (un_subregion) - Unique values:")
print("-" * 80)
regions = df['un_subregion'].value_counts(dropna=False)
for region, count in regions.items():
    print(f"  {region}: {count:,} ({count/len(df)*100:.1f}%)")

## 4. Define Target Languages

Filter for entities with Wikipedia articles in **all five** languages: en (English), de (German), fr (French), it (Italian), es (Spanish).

In [ ]:
# Generate Wikipedia edition format (e.g., 'enwiki', 'itwiki')
# This format is used in the list_wikipedia_editions column
required_languages_wiki = [f"{lang}wiki" for lang in REQUIRED_LANGUAGES]

print(f'Base language codes: {REQUIRED_LANGUAGES}')
print(f'Wikipedia editions: {required_languages_wiki}')

## 5. Filter by Required Languages

Keep only entities with articles in **all 5 languages** (multilayer network requirement).

In [ ]:
# Function to check if all required languages are present in the entry
def has_required_languages(lang_list_str: str, required: List[str]) -> bool:
    langs = set(lang_list_str.split('|'))
    return all(lang in langs for lang in required)

# Apply the filter using Wikipedia edition format
df_filtered = df[df['list_wikipedia_editions'].apply(
    lambda x: has_required_languages(x, required_languages_wiki)
)]

print(f"Rows before filtering: {len(df):,}")
print(f"Rows after filtering: {len(df_filtered):,}")
print(f"Retention rate: {len(df_filtered)/len(df)*100:.1f}%")

## 6. Preview Filtered Data

In [ ]:
# Display the first few rows of the filtered DataFrame
df_filtered.head()

# Optionally, save the filtered DataFrame to a new CSV file
# df_filtered.to_csv('../data/cross-verified-database_filtered.csv.gz', index=False, compression='gzip')

## 7. Save Selected Columns

In [ ]:
# Create final filtered dataset with selected columns
selected_columns = ['wikidata_id', 'birth', 'bigperiod_birth', 'un_subregion', 'gender']
df_final = df_filtered[selected_columns].copy()

print(f"Selected columns: {selected_columns}")
print(f"\nFinal dataset: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print("\nSample:")
print("=" * 80)
print(df_final.head(10))
print("\n")

# Save to CSV
df_final.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved to: {OUTPUT_FILE}")

## Summary

**Completed**: Entity filtering for multilayer network analysis

- **Filter**: Entities with articles in all 5 languages (en, it, fr, es, de)
- **Output**: `entities_filtered_by_languages.csv` with 4 columns
- **Next**: Graph construction with `python src/main.py build`

See [README.md](../README.md) for full pipeline documentation.